# Phase 1 — Data acquisition and inspection

We are not using TxGNN's `TxData` helper (dropped in Phase 0, see `README.md`), so PrimeKG is downloaded directly from Harvard Dataverse (Chandak et al., *Scientific Data* 2023, DOI: 10.7910/DVN/IXA7BM).

This notebook:
1. Queries the Dataverse API for the dataset's file list (no hardcoded file IDs — those can silently go stale).
2. Downloads the main KG edge-list file.
3. Loads it and **prints the real counts** — nothing here is hardcoded from the paper.
4. Saves `data_card.json` with those real numbers.

Mounts Google Drive this time, since the downloaded file is large enough (PrimeKG's edge list is several hundred MB) that re-downloading every session would be wasteful.

In [18]:
from google.colab import drive
drive.mount('/content/drive')
import os
PROJECT_DIR = '/content/drive/MyDrive/txgnn_comparison'
for sub in ['data', 'checkpoints', 'results']:
    os.makedirs(f'{PROJECT_DIR}/{sub}', exist_ok=True)
print('Project dir:', PROJECT_DIR)

Mounted at /content/drive
Project dir: /content/drive/MyDrive/txgnn_comparison


In [24]:
# 1.1a Query Dataverse for the real file list (don't guess file IDs)
import requests, json

DOI = 'doi:10.7910/DVN/IXA7BM'
resp = requests.get(
    'https://dataverse.harvard.edu/api/datasets/:persistentId/',
    params={'persistentId': DOI}
)
resp.raise_for_status()
meta = resp.json()
files = meta['data']['latestVersion']['files']
for f in files:
    df = f['dataFile']
    print(df['id'], df['filename'], df.get('filesize'), df.get('contentType'))

6180618 disease_features.tab 113534270 text/tab-separated-values
6180619 drug_features.tab 10030011 text/tab-separated-values
6180616 edges.csv 386582390 text/csv
6180620 kg.csv 981751236 text/csv
6180625 kg_giant.csv 893328974 text/csv
6180626 kg_grouped.csv 888336264 text/csv
6180623 kg_grouped_diseases_bert_map.tab 1402368 text/tab-separated-values
6180624 kg_grouped_diseases.tab 2915442 text/tab-separated-values
6180622 kg_raw.csv 893347412 text/csv
6180617 nodes.tab 8893757 text/tab-separated-values
6191270 README.txt 3137 text/plain


In [20]:
# 1.1b Pick the main KG edge-list file from the real listing above
# (PrimeKG's full edge list is conventionally named kg.csv — confirm against
# the printed filenames in the previous cell; adjust the match string if the
# actual filename differs).
kg_file = next(f['dataFile'] for f in files if f['dataFile']['filename'].lower() == 'kg.csv')
kg_file_id = kg_file['id']
print('Selected file:', kg_file['filename'], 'id=', kg_file_id, 'size=', kg_file.get('filesize'))

kg_path = f"{PROJECT_DIR}/data/kg.csv"
if not os.path.exists(kg_path):
    url = f'https://dataverse.harvard.edu/api/access/datafile/{kg_file_id}'
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(kg_path, 'wb') as out:
            for chunk in r.iter_content(chunk_size=1 << 20):
                out.write(chunk)
    print('Downloaded to', kg_path)
else:
    print('Already present at', kg_path, '— skipping re-download')

print('File size on disk (bytes):', os.path.getsize(kg_path))

Selected file: kg.csv id= 6180620 size= 981751236
Downloaded to /content/drive/MyDrive/txgnn_comparison/data/kg.csv
File size on disk (bytes): 981751236


In [21]:
# 1.2 Inspect the graph — every number below is computed from the actual
# downloaded file, not recalled from the paper.
import pandas as pd

kg = pd.read_csv(kg_path, low_memory=False)
print('Shape:', kg.shape)
print('Columns:', list(kg.columns))
kg.head()

Shape: (8100498, 12)
Columns: ['relation', 'display_relation', 'x_index', 'x_id', 'x_type', 'x_name', 'x_source', 'y_index', 'y_id', 'y_type', 'y_name', 'y_source']


,relation,display_relation,x_index,x_id,x_type,x_name,x_source,y_index,y_id,y_type,y_name,y_source
0,protein_protein,ppi,0,9796,gene/protein,PHYHIP,NCBI,8889,56992,gene/protein,KIF15,NCBI
1,protein_protein,ppi,1,7918,gene/protein,GPANK1,NCBI,2798,9240,gene/protein,PNMA1,NCBI
2,protein_protein,ppi,2,8233,gene/protein,ZRSR2,NCBI,5646,23548,gene/protein,TTC33,NCBI
3,protein_protein,ppi,3,4899,gene/protein,NRF1,NCBI,11592,11253,gene/protein,MAN1B1,NCBI
4,protein_protein,ppi,4,5297,gene/protein,PI4KA,NCBI,2122,8601,gene/protein,RGS20,NCBI


In [22]:
# Expected PrimeKG schema: relation, display_relation, x_index, x_id, x_type,
# x_name, x_source, y_index, y_id, y_type, y_name, y_source.
# If the columns printed above differ, fix the column names referenced below
# before trusting these counts.
node_cols_x = kg[['x_index', 'x_id', 'x_type', 'x_name']].rename(
    columns={'x_index': 'index', 'x_id': 'id', 'x_type': 'type', 'x_name': 'name'})
node_cols_y = kg[['y_index', 'y_id', 'y_type', 'y_name']].rename(
    columns={'y_index': 'index', 'y_id': 'id', 'y_type': 'type', 'y_name': 'name'})
all_nodes = pd.concat([node_cols_x, node_cols_y]).drop_duplicates(subset=['index'])

node_counts_by_type = all_nodes['type'].value_counts().to_dict()
edge_counts_by_relation = kg['relation'].value_counts().to_dict()

print('Total unique nodes:', len(all_nodes))
print('Node counts by type:')
for t, c in node_counts_by_type.items():
    print(' ', t, c)
print('Total edges:', len(kg))
print('Edge counts by relation:')
for r, c in edge_counts_by_relation.items():
    print(' ', r, c)

Total unique nodes: 129375
Node counts by type:
  biological_process 28642
  gene/protein 27671
  disease 17080
  effect/phenotype 15311
  anatomy 14035
  molecular_function 11169
  drug 7957
  cellular_component 4176
  pathway 2516
  exposure 818
Total edges: 8100498
Edge counts by relation:
  anatomy_protein_present 3036406
  drug_drug 2672628
  protein_protein 642150
  disease_phenotype_positive 300634
  bioprocess_protein 289610
  cellcomp_protein 166804
  disease_protein 160822
  molfunc_protein 139060
  drug_effect 129568
  bioprocess_bioprocess 105772
  pathway_protein 85292
  disease_disease 64388
  contraindication 61350
  drug_protein 51306
  anatomy_protein_absent 39774
  phenotype_phenotype 37472
  anatomy_anatomy 28064
  molfunc_molfunc 27148
  indication 18776
  cellcomp_cellcomp 9690
  phenotype_protein 6660
  off-label use 5136
  pathway_pathway 5070
  exposure_disease 4608
  exposure_exposure 4140
  exposure_bioprocess 3250
  exposure_protein 2424
  disease_phenotype_n

In [23]:
# Save data_card.json with the real numbers computed above
import datetime

data_card = {
    'kg_name': 'PrimeKG',
    'source': 'Chandak et al., Scientific Data 2023, DOI: 10.7910/DVN/IXA7BM',
    'downloaded_file': kg_file['filename'],
    'file_size_bytes': os.path.getsize(kg_path),
    'total_nodes': int(len(all_nodes)),
    'node_counts_by_type': {k: int(v) for k, v in node_counts_by_type.items()},
    'total_edges': int(len(kg)),
    'edge_counts_by_relation': {k: int(v) for k, v in edge_counts_by_relation.items()},
    'disease_nodes': int(node_counts_by_type.get('disease', 0)),
    'drug_nodes': int(node_counts_by_type.get('drug', 0)),
    'retrieved_date': datetime.date.today().isoformat(),
    'notes': 'All counts computed directly from the downloaded kg.csv, not copied from the paper.'
}

with open(f'{PROJECT_DIR}/results/data_card.json', 'w') as f:
    json.dump(data_card, f, indent=2)

print(json.dumps(data_card, indent=2))

{
  "kg_name": "PrimeKG",
  "source": "Chandak et al., Scientific Data 2023, DOI: 10.7910/DVN/IXA7BM",
  "downloaded_file": "kg.csv",
  "file_size_bytes": 981751236,
  "total_nodes": 129375,
  "node_counts_by_type": {
    "biological_process": 28642,
    "gene/protein": 27671,
    "disease": 17080,
    "effect/phenotype": 15311,
    "anatomy": 14035,
    "molecular_function": 11169,
    "drug": 7957,
    "cellular_component": 4176,
    "pathway": 2516,
    "exposure": 818
  },
  "total_edges": 8100498,
  "edge_counts_by_relation": {
    "anatomy_protein_present": 3036406,
    "drug_drug": 2672628,
    "protein_protein": 642150,
    "disease_phenotype_positive": 300634,
    "bioprocess_protein": 289610,
    "cellcomp_protein": 166804,
    "disease_protein": 160822,
    "molfunc_protein": 139060,
    "drug_effect": 129568,
    "bioprocess_bioprocess": 105772,
    "pathway_protein": 85292,
    "disease_disease": 64388,
    "contraindication": 61350,
    "drug_protein": 51306,
    "anatomy